# Stage 2 Combined Preprocessing (ISIC 2019 + PAD-UFES-20)

This notebook prepares **combined Stage-2 splits** for ISIC 2019 and PAD-UFES-20, with HAM10000 MD5 deduplication applied to ISIC records before split.

In [1]:
import importlib
import skin_pipeline.datasets as ds

importlib.reload(ds)
build_isic_records = ds.build_isic_records  # refresh local reference

c:\Users\prakh\anaconda3\envs\dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import hashlib
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

from skin_pipeline.datasets import build_isic_records, build_pad_records

SEED = 42
np.random.seed(SEED)

DATA_ROOT = Path('data')
HAM_TRAIN_CSV = DATA_ROOT / 'HAM10000' / 'ham_train.csv'

ISIC_GT_CSV = DATA_ROOT / 'isic' / 'ISIC_2019_Training_GroundTruth.csv'
ISIC_META_CSV = DATA_ROOT / 'isic' / 'ISIC_2019_Training_Metadata.csv'
ISIC_IMAGE_DIR = DATA_ROOT / 'isic' / 'ISIC_2019_Training_Input'

PAD_META_CSV = DATA_ROOT / 'PAD-UFES-20' / 'archive' / 'metadata.csv'
PAD_IMAGES_ROOT = DATA_ROOT / 'PAD-UFES-20' / 'archive'

OUTPUT_DIR = DATA_ROOT / 'stage2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output dir:', OUTPUT_DIR)

Output dir: data\stage2


In [3]:
def md5_file(path: str, chunk_size: int = 1024 * 1024) -> str | None:
    p = Path(path)
    if not p.exists():
        return None
    h = hashlib.md5()
    with open(p, 'rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

In [4]:
# 1) Build raw records
isic_records = build_isic_records(str(ISIC_GT_CSV), str(ISIC_META_CSV), str(ISIC_IMAGE_DIR))
pad_records = build_pad_records(str(PAD_META_CSV), str(PAD_IMAGES_ROOT))

print('Raw ISIC records:', len(isic_records))
print('Raw PAD records:', len(pad_records))

Raw ISIC records: 25331
Raw PAD records: 2298


In [6]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

CACHE_CSV = Path("data/stage2/md5_cache.csv")
CACHE_CSV.parent.mkdir(parents=True, exist_ok=True)

cache = {}
if CACHE_CSV.exists():
    cdf = pd.read_csv(CACHE_CSV)
    cache = dict(zip(cdf["path"], cdf["md5"]))

def md5_cached(p):
    p = str(p)
    if p in cache:
        return cache[p]
    h = md5_file(p)
    cache[p] = h
    return h

ham_paths = pd.read_csv(HAM_TRAIN_CSV)["image_path"].astype(str).tolist()
with ThreadPoolExecutor(max_workers=8) as ex:
    ham_hashes = set(h for h in ex.map(md5_cached, ham_paths) if h)

with ThreadPoolExecutor(max_workers=8) as ex:
    isic_hashes = list(ex.map(lambda r: md5_cached(r[0]), isic_records))

isic_dedup, removed = [], 0
for rec, h in zip(isic_records, isic_hashes):
    if h is not None and h in ham_hashes:
        removed += 1
    else:
        isic_dedup.append(rec)

pd.DataFrame({"path": list(cache.keys()), "md5": list(cache.values())}).to_csv(CACHE_CSV, index=False)

print("HAM hashes:", len(ham_hashes))
print("ISIC removed via dedup:", removed)
print("ISIC after dedup:", len(isic_dedup))

HAM hashes: 7053
ISIC removed via dedup: 7054
ISIC after dedup: 18277


In [7]:
# 3) Convert to DataFrames
isic_df = pd.DataFrame(isic_dedup, columns=['image_path', 'label', 'source', 'fst_group', 'patient_id'])
pad_df = pd.DataFrame(pad_records, columns=['image_path', 'label', 'source', 'fst_group', 'patient_id'])

# Ensure patient ids are strings for splitting
isic_df['patient_id'] = isic_df['patient_id'].astype(str)
pad_df['patient_id'] = pad_df['patient_id'].astype(str)

print(isic_df.head(2))
print(pad_df.head(2))

                                          image_path label    source  \
0  data\isic\ISIC_2019_Training_Input\ISIC_2019_T...    NV  ISIC2019   
1  data\isic\ISIC_2019_Training_Input\ISIC_2019_T...    NV  ISIC2019   

   fst_group patient_id  
0          0        nan  
1          0        nan  
                                          image_path label       source  \
0  data\PAD-UFES-20\archive\imgs_part_3\imgs_part...   nev  PAD-UFES-20   
1  data\PAD-UFES-20\archive\imgs_part_1\imgs_part...   bcc  PAD-UFES-20   

   fst_group patient_id  
0          0   PAT_1516  
1          0     PAT_46  


In [8]:
def grouped_train_val_test_split(df: pd.DataFrame, group_col: str = 'patient_id', seed: int = 42):
    # 70/15/15 by grouped splitting
    gss1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=seed)
    idx = np.arange(len(df))
    train_idx, temp_idx = next(gss1.split(idx, groups=df[group_col]))

    temp_df = df.iloc[temp_idx].copy()
    gss2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=seed)
    temp_idx2 = np.arange(len(temp_df))
    val_rel_idx, test_rel_idx = next(gss2.split(temp_idx2, groups=temp_df[group_col]))

    train_df = df.iloc[train_idx].copy()
    val_df = temp_df.iloc[val_rel_idx].copy()
    test_df = temp_df.iloc[test_rel_idx].copy()
    return train_df, val_df, test_df

isic_train, isic_val, isic_test = grouped_train_val_test_split(isic_df)
pad_train, pad_val, pad_test = grouped_train_val_test_split(pad_df)

print('ISIC split sizes:', len(isic_train), len(isic_val), len(isic_test))
print('PAD split sizes :', len(pad_train), len(pad_val), len(pad_test))

ISIC split sizes: 13255 2494 2528
PAD split sizes : 1582 370 346


In [9]:
# 4) Combine splits for Stage-2 training/eval
combined_train = pd.concat([isic_train, pad_train], ignore_index=True)
combined_val = pd.concat([isic_val, pad_val], ignore_index=True)
combined_test = pd.concat([isic_test, pad_test], ignore_index=True)

# Shuffle each split (deterministic)
combined_train = combined_train.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
combined_val = combined_val.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
combined_test = combined_test.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print('Combined split sizes:', len(combined_train), len(combined_val), len(combined_test))
print('Combined source distribution (train):')
print(combined_train['source'].value_counts())

Combined split sizes: 14837 2864 2874
Combined source distribution (train):
source
ISIC2019       13255
PAD-UFES-20     1582
Name: count, dtype: int64


In [11]:
# 5) Save all useful CSV artifacts
isic_train.to_csv(OUTPUT_DIR / 'isic_stage2_train.csv', index=False)
isic_val.to_csv(OUTPUT_DIR / 'isic_stage2_val.csv', index=False)
isic_test.to_csv(OUTPUT_DIR / 'isic_stage2_test.csv', index=False)

pad_train.to_csv(OUTPUT_DIR / 'pad_stage2_train.csv', index=False)
pad_val.to_csv(OUTPUT_DIR / 'pad_stage2_val.csv', index=False)
pad_test.to_csv(OUTPUT_DIR / 'pad_stage2_test.csv', index=False)

combined_train.to_csv(OUTPUT_DIR / 'stage2_combined_train.csv', index=False)
combined_val.to_csv(OUTPUT_DIR / 'stage2_combined_val.csv', index=False)
combined_test.to_csv(OUTPUT_DIR / 'stage2_combined_test.csv', index=False)

print('Saved stage-2 preprocessing CSVs to:', OUTPUT_DIR)

Saved stage-2 preprocessing CSVs to: data\stage2


In [12]:
# 6) Leakage sanity checks per source

def assert_no_group_overlap(a: pd.DataFrame, b: pd.DataFrame, name_a: str, name_b: str):
    overlap = set(a['patient_id'].astype(str)).intersection(set(b['patient_id'].astype(str)))
    assert len(overlap) == 0, f'Patient leakage between {name_a} and {name_b}: {len(overlap)}'

assert_no_group_overlap(isic_train, isic_val, 'isic_train', 'isic_val')
assert_no_group_overlap(isic_train, isic_test, 'isic_train', 'isic_test')
assert_no_group_overlap(isic_val, isic_test, 'isic_val', 'isic_test')

assert_no_group_overlap(pad_train, pad_val, 'pad_train', 'pad_val')
assert_no_group_overlap(pad_train, pad_test, 'pad_train', 'pad_test')
assert_no_group_overlap(pad_val, pad_test, 'pad_val', 'pad_test')

print('Patient/group leakage checks passed.')

Patient/group leakage checks passed.
